<a href="https://colab.research.google.com/github/FarazTheAnalyst/Data-Scientist-Portfolio/blob/main/Resume%20Screening%20with%20RAG%20%2B%20LLM%20Project%20/rag_resume_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install faiss-cpu
!pip install PyPDF2 python-docx

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import pickle
import faiss
from sentence_transformers import SentenceTransformer
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import PyPDF2
import docx
import re
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# Load dataset
df = pd.read_csv("/content/AI_Resume_Matcher_Dataset.csv")
print(f"Loaded {len(df)} resumes")
print(f"Columns: {df.columns.tolist()}")
print(f"Sample data:\n{df.head(2)}")
print(f"\nDataset info:")
print(df.info())

In [ ]:
# Check for missing values
print(f"\nMissing values:\n{df.isnull().sum()}")

TEXT EXTRACTION FUNCTIONS (CRITICAL FOR HANDLING DIFFERENT FILE TYPES)

In [ ]:
# Text extraction from different formats
def extract_text_from_file(file_path):
  if file_path.endswith(".pdf"):
    with open(file_path, "rb") as file:
      reader = PyPDF2.PdfReader(file)  #read pages and text
      text = ""
      for page in reader.pages:
        text += page.extract_text()
    return text
  elif file_path.endswith(".docx"):
    doc = docx.Document(file_path)
    text = "\n".join([paragraph.text for paragraph in doc.paragraphs])
    return text
  else:
      with open(file_path, "r", encoding="utf-8", errors="ignore") as file:
        return file.read()

TEXT PREPROCESSING FUNCTIONS

In [ ]:
# Preprocess and create resume text from structured data
def create_resume_text(row):
  """Create a comprehensive resume text from structured data"""
  resume_parts = []

  # Name
  if pd.notna(row.get("Name", "")):
    resume_parts.append(f"Name: {row["Name"]}")

  # Experience
  if pd.notna(row.get("Experience_Years", "")):
    resume_parts.append(f"Experience: {row['Experience_Years']} Years")

  # Skills
  if pd.notna(row.get("Skills", "")):
    skills = row["Skills"]
    if isinstance(skills, str):
      # Split skills if they're comma-separated
      skill_list = [s.strip() for s in skills.split(",")]
      resume_parts.append(f"Skills: {', '.join(skill_list)}")

  # Education
  if pd.notna(row.get("Education", "")):
    resume_parts.append(f"Education: {row["Education"]}")

  # Applied job Role
  if pd.notna(row.get("Applied_Job_Role", "")):
    resume_parts.append(f"Applied for: {row['Applied_Job_Role']}")

  return " ".join(resume_parts) if resume_parts else "No resume data available"

In [ ]:
# Create processed resumes
df["Resume_Text"] = df.apply(create_resume_text, axis=1)

In [ ]:
# Preprocess resume text
def preprocess_text(text):
  # remove extra whitepace
  text = re.sub(r"\s", " ", text).strip()

  # Remove special characters (keep only basic punctuation)
  text = re.sub(r"[^\w\s\.\,\-\:]", " ", text)

  return text

In [ ]:
df["Processed_Text"] = df["Resume_Text"].apply(preprocess_text)

In [ ]:
# create resume chunks for better retrieval
def chunk_resume(text, chunk_size=500, ovarlap=50):
  """
  Split resume text into overlaping chunks for better retrieval
  """
  if not text or len(text.strip()) == 0:
    return ["No content available"]

  words = text.split()
  chunks = []

  if len(words) == 0:
    return [text]

  for i in range(0, len(words), chunk_size - ovarlap):
    chunk = " ".join(words[i:i + chunk_size])
    if chunk.strip():
      chunks.append(chunk)
  # if no chunk were created, return the original text
  if not chunks:
    chunks = [text[:500]]

  return chunks

EMBEDDING MODEL

In [ ]:
# Initialize embedding model
embedding_model = SentenceTransformer("all-MiniLm-L6-v2")
embedding_dim = embedding_model.get_sentence_embedding_dimension()
print(f"Embedding dimension: {embedding_dim}")

VECTOR DATABASE WITH FAISS

PROCESS RESUMES FROM DATASET:

In [ ]:
# Create vector database with FAISS
def create_vector_database(resume_data, embedding_model):
  all_chunks = []
  all_metadata = []

  for resume in resume_data:
    for chunk_idx, chunk in enumerate(resume["chunks"]):
      all_chunks.append(chunk)
      all_metadata.append({
          "resume_id": resume["id"],
          "chunk_idx": chunk_idx,
          "name": resume["name"],
          "experience_years": resume["experience_years"],
          "skills": resume["skills"],
          "education": resume["education"],
          "applied_job_role": resume["applied_job_role"],
          "full_text": resume["full_text"][:300] + "..." if len(resume["full_text"]) > 300 else resume["full_text"]
      })
  # Generate embeddings
  print("Generating embeddings...")
  embeddings = embedding_model.encode(all_chunks, show_progress_bar=True)
  embeddings = np.array(embeddings).astype("float32")

  # Create FAISS index
  index = faiss.IndexFlatL2(embedding_dim) #Index is created.No vectors have been added yet. L2 Eulidean distance
  index.add(embeddings)  #vectors are stored

  return index, all_chunks, all_metadata

# --- Resume data creation (moved here for robustness) ---
resume_data = []
for idx, row in tqdm(df.iterrows(), total=len(df)):
    # Extract and preprocess text
    resume_text = row['Processed_Text']

    # Create chunks
    chunks = chunk_resume(resume_text)

    resume_data.append({
        'id': idx,
        'name': row.get('Name', f'Candidate_{idx}'),
        'experience_years': row.get('Experience_Years', 0),
        'skills': row.get('Skills', ''),
        'education': row.get('Education', ''),
        'applied_job_role': row.get('Applied_Job_Role', ''),
        'full_text': resume_text,
        'chunks': chunks,
        'num_chunks': len(chunks),
        'metadata': {
            'name': row.get('Name', f'Candidate_{idx}'),
            'experience_years': row.get('Experience_Years', 0),
            'skills': row.get('Skills', ''),
            'education': row.get('Education', ''),
            'applied_job_role': row.get('Applied_Job_Role', ''),
            'file_name': f"resume_{idx}.txt"
        }
    })

print(f"\nProcessed {len(resume_data)} resumes with {sum(r['num_chunks'] for r in resume_data)} total chunks")
# --- End of resume data creation ---

# Create vector database
vector_index, chunks, metadata = create_vector_database(resume_data, embedding_model)
print(f"Created vector database with {len(chunks)} chunks")

In [ ]:
resume_data[0]

{'id': 0,
 'name': 'Candidate_0',
 'experience_years': 1,
 'skills': 'Node.js, MongoDB',
 'education': 'BCA',
 'applied_job_role': 'Data Scientist',
 'full_text': 'Name: Candidate_0 Experience: 1 Years Skills: Node.js, MongoDB Education: BCA Applied for: Data Scientist',
 'chunks': ['Name: Candidate_0 Experience: 1 Years Skills: Node.js, MongoDB Education: BCA Applied for: Data Scientist'],
 'num_chunks': 1,
 'metadata': {'name': 'Candidate_0',
  'experience_years': 1,
  'skills': 'Node.js, MongoDB',
  'education': 'BCA',
  'applied_job_role': 'Data Scientist',
  'file_name': 'resume_0.txt'}}

In [ ]:
# Save vector database
faiss.write_index(vector_index, "vector_database.index")
with open("chunks.pkl", "wb") as f:
  pickle.dump((chunks, metadata), f)

print("Vecto database saved successfullyr")

Vecto database saved successfullyr


**LLM** FOR **RAG**

In [ ]:
# Initialize LLM for RAG
class ResumeRAG:
    def __init__(self, model_name="microsoft/DialoGPT-medium"):
      self.tokenizer = AutoTokenizer.from_pretrained(model_name)
      self.model = AutoModelForCausalLM.from_pretrained(model_name)
      self.model.to(device)
      self.model.eval()

      self.embedding_model = embedding_model
      self.vector_index = vector_index
      self.chunks = chunks
      self.metadata = metadata

    def retrieve_relevant_chunks(self, query, top_k=5):
      # Generate query embedding
      query_embedding = self.embedding_model.encode([query])[0].astype("float32")
      # Search in FAISS
      distances, indices = self.vector_index.search(query_embedding.reshape(1, -1), top_k)
      # Get relevant chunks
      relevant_chunks = []
      for idx in indices[0]:
        if idx < len(self.chunks):
          relevant_chunks.append({
              "chunk": self.chunks[idx][:500] + "...",
              "metadata": self.metadata[idx],
              "distance": float(distances[0][list(indices[0]).index(idx)])
          })
      return relevant_chunks

    def generate_evaluation(self, job_description, resume_text):
        # Retrieve relevant chunks
        relevant_chunks = self.retrieve_relevant_chunks(job_description, top_k=5)

        # Contruct prompt
        prompt = f"""
          Job Description: {job_description[:1000]}
          Candidate Resume: {resume_text[:1500]}

          Relevant Experience from database:
          {chr(10).join([chunk["chunk"] for chunk in relevant_chunks[:3]])}

          Based on the job description and candidate resume, provide the following evaluation

          1. Overall Match Score (9, 199)
          2. key Strengths (list 3-5)
          3. key Weekneses (list 2-3)
          4. Specific Skills (list)
          5. Missing Skills(list)
          6. Overall Recommendation (yes/no)

          Evaluation:
        """

        # Generate response
        inputs = self.tokenizer(prompt, return_tensors="pt", maxt_length=1000, trunction=True).to(device)

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.inference_mode():
          outputs = self.model.generate(
              **inputs,
              max_new_tokens=500,
              temperature=0.7,
              do_sample=True,
              pad_token_id=self.tokenizer.eos_token_id
          )

        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Extract evaluation from response
        try:
            evaluation_text = response.split("Evaluation:")[1].strip()
        except:
            evaluation_text = response

        return {
            "evaluation": evaluation_text,
            "relevant_chunks": relevant_chunks
        }

# Initialize RAG system
rag_system = ResumeRAG()


SAVE RAG SYSTEM

In [ ]:
with open("rag_system.pkl", "wb") as f:
  pickle.dump({
      "vector_index": vector_index,
      "chunks": chunks,
      "metadata": metadata,
      "embedding_model": embedding_model
  }, f)

print("RAG system saved successfully")

TESTING RAG SYSTEM

In [ ]:
#Test the RAG system
test_job = """
We are looking for a Data Scientist with expertise in machine learning, Python, and data
analysis.
The candidate should have experience with statistical modeling and data visualization
"""
test_candidate = resume_data[0]["full_text"]
evaluation = rag_system.generate_evaluation(test_job, test_candidate)

print("\n=== Test Evaluation ===")
print(evaluation["evaluation"])
print("\n=== Relevant Chunks ===")
for chunk in evaluation["relevant_chunks"][:3]:
  print(f"- {chunk['metadata']['name']} ({chunk['metadata']['applied_job_role']}): {chunk['chunk'][:100]}...")

SAMPLE VISUALIZATION

In [ ]:
# Create sample visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of experience
plt.figure(figsize=(12, 6))
df["Experience_Years"].hist(bins=20, edgecolor="black")
plt.title(Experience Distribution of Candidate)
plt.xlabel("Years of Experience")
plt.ylabel("Number of Candidates")
plt.tight_layout()
plt.savefig("experiecne_deistribution.png")
plt.close()

In [ ]:
# Ditribution of applied roles
if "Applied_Job_Role" in df.columns:
  plt.figure(figsize=(14, 6))
  df["Applied_Job_Role"].value_counts().head(10).plot(kind="bar")
  plt.title("Top 10 Applied Job Roles")
  plt.xlabel("Job Role")
  plt.ylabel("Number of Candidates")
  plt.xticks(rotation=45)
  plt.tight_layout()
  plt.savefig("applied_roles.png")
  plt.close()
else:
  print("No 'Applied_Job_Role' column found in the DataFrame.")

In [ ]:
# Education distribution
if "Educationn" in df.columns:
  plt.figure(figsize=(12, 6))
  df["Education"].value_counts().plot(kind="bar")
  plt.title("Education Dsitribution")
  plt.xlabel("Education Level")
  plt.ylabel("Number of candidates")
  plt.xticks(rotation=45)
  plt.tight_layout()
  plt.savefig("education_distribution.png")
  plt.close()
else:
    print("No 'Education' column found in the DataFrame.")